In [3]:
%pip install faiss-cpu sentence-transformers groq pypdf dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
import dotenv
import groq
from pypdf import PdfReader

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
docs = []

folder = "D:\\KCE\\aiml - training\\LLM\\RESUMES"  # Update this path to your folder containing the PDF resumes

for file in os.listdir(folder):
    if file.endswith(".pdf"):
        reader = PdfReader(os.path.join(folder, file))
        
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        
        docs.append(text)

print("Total resumes:", len(docs))

Total resumes: 32


In [7]:
doc_embeddings = model.encode(docs).astype("float32")

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print("FAISS index created")

FAISS index created


In [8]:
query = "Which candidate knows suitable for AI engineer position?"

query_embedding = model.encode([query]).astype("float32")

top_k = 3

distance, idx = index.search(query_embedding, k=top_k)

print("Distances:", distance)
print("Indexes:", idx)

Distances: [[0.94214624 1.0040107  1.0049982 ]]
Indexes: [[17 26  3]]


In [9]:
retrieved_chunks = [docs[i] for i in idx[0]]

for r in retrieved_chunks:
    print("-"*50)
    print(r[:500])

--------------------------------------------------
MADHAN R 
Student – B.Tech Artificial Intelligence and Data Science 
Karpagam College of Engineering
  
 + 91 6380642697        |  
mathansmart30@gmail.com
 
       www.linkedin.com/in/madhan-r-37834029a 
 
      https://github.com/madhan 
CAREER OBJECTIVE 
Driven by curiosity and innovation, I aim to harness the power of AI and data to uncover insights, build intelligent 
systems, and shape smarter, more efficient solutions that benefit industries and society alike.  To obtain a 
challenging po
--------------------------------------------------
 
     POOJASRI M 
     II year 
       (+91) 8056049315     poojasrinirmalamanickam@gmail.com 
            www.linkedin.com/in/poojasri-nirmalamanickam-413269215/    https://github.com/Poojasri37 
Profile Summary 
Aspiring AI & Data Science engineer with hands -on experience in machine learning, MLOps, and Python 
development. Passionate about building impactful applications through data -driv

In [10]:
client = groq.Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

context = "\n\n".join(retrieved_chunks)

prompt = f"""
Use the following resumes to answer the question.

Resumes:
{context}

Question:
{query}
"""
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="llama-3.1-8b-instant",
)

print(chat_completion.choices[0].message.content)

Based on the given resumes, I would recommend Poojasri M for the AI engineer position. Here's why:

1. **Experience**: Poojasri has significant internship experience in three tech domains: TechnoHacks EduTech, Prodigy Infotech, and BYTS. Her experience in machine learning, data science, and Python development is directly relevant to the AI engineer role.
2. **Skills**: Her technical skills match the requirements of an AI engineer, including proficiency in Python, machine learning frameworks (e.g., Scikit-learn, Keras, TensorFlow), and data analysis tools (e.g., Pandas, Matplotlib).
3. **Projects**: She has worked on several projects that demonstrate her expertise in AI, including building and optimizing ML models, developing datasets, and deploying models using various frameworks.
4. **Certifications**: Poojasri has obtained relevant certifications in Python for Data Science, Business Analyst Qualification, and Machine Learning, which indicate her dedication to learning and staying up-